This file constructs the dataset of newspaper articles from New York Times, Wall Street Journal, and Washington Post from 1980/01/01 to 2025/12/31.

In [1]:
# Loaded libraries
library(xml2);       # XML processing
library(rvest);      # Cleanly strip HTML from document text
library(parallel);   # Multi-core processing
library(data.table); # Fast reading/writing datas
library(dplyr)

if (!dir.exists("../output_files/")) {
    dir.create("../output_files/")
}


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
# Create list of xml files
create_xml <- function(dataset_name){
    xml_files <- list.files(paste0('/home/ec2-user/SageMaker/data/',dataset_name), pattern = "*xml$",full.names = TRUE)
    print(paste("Files in dataset", length(xml_files)))
    return(xml_files)
    }

In [4]:
NYT_1_xml <- create_xml("NYT19801996")  ## Note that these are dataset names created from ProQuest TDM Studio
NYT_2_xml <- create_xml("NYT19972021")
NYT_3_xml <- create_xml("NYT20222025")

[1] "Files in dataset 1970856"
[1] "Files in dataset 1972543"
[1] "Files in dataset 151781"


In [5]:
head(NYT_1_xml)

[1] "/home/ec2-user/SageMaker/data/NYT19801996/1095348055.xml"
[2] "/home/ec2-user/SageMaker/data/NYT19801996/1114792815.xml"
[3] "/home/ec2-user/SageMaker/data/NYT19801996/1114792819.xml"
[4] "/home/ec2-user/SageMaker/data/NYT19801996/1524225262.xml"
[5] "/home/ec2-user/SageMaker/data/NYT19801996/1790141900.xml"
[6] "/home/ec2-user/SageMaker/data/NYT19801996/1790142054.xml"

In [6]:
WP_1_xml <- create_xml("WP19872010")
WP_2_xml <- create_xml("WP20112025")

[1] "Files in dataset 1975776"
[1] "Files in dataset 1659051"


In [7]:
WSJ_xml <- create_xml("WSJ")

[1] "Files in dataset 1726842"


In [3]:
extract <- function(filename){
    
    # F.1
    # Read a single XML document
    doc_xml <- read_xml(filename)
    
    # F.2
    # Find and extract desired data
    Article_ID  <- xml_find_all(doc_xml, "//GOID") %>% xml_text() %>% paste0(collapse = "; ")
    Title       <- xml_find_all(doc_xml, "//TitleAtt/Title") %>% xml_text() %>% paste0(collapse = "; ")
    Date        <- xml_find_all(doc_xml, "//NumericDate") %>% xml_text() %>% paste0(collapse = "; ")
    Location    <- xml_find_all(doc_xml, "//Term/Geographic") %>% xml_text() %>% paste0(collapse = "; ")
    People      <- xml_find_all(doc_xml, "//Term/Personal") %>% xml_text() %>% paste0(collapse = "; ")
    Organization <- xml_find_all(doc_xml, "//Terms/CompanyTerm/CompanyNameAtt/CompanyName") %>% xml_text() %>% paste0(collapse = "; ")
    Type        <- xml_find_all(doc_xml, "//mstar") %>% xml_text() %>% paste0(collapse = "; ")
    Source      <- xml_find_all(doc_xml, "//DFS/PubFrosting/Title") %>% xml_text() %>% paste0(collapse = "; ")
    Desk        <- xml_find_all(doc_xml, "//Desk") %>% xml_text() %>% paste0(collapse = "; ")

    # F.3
    # For Abstract and Text, check length before attempting HTML parsing
    abstract_raw <- xml_find_all(doc_xml, "//Abstract/Short") %>% xml_text()
    Abstract <- if (length(abstract_raw) == 0) NA_character_ else paste0(abstract_raw, collapse = " ") %>% read_html() %>% html_text()

    text_raw <- xml_find_all(doc_xml, "//TextInfo/HiddenText") %>% xml_text()
    Text <- if (length(text_raw) == 0) NA_character_ else paste0(text_raw, collapse = " ") %>% read_html() %>% html_text()
    
    # F.4
    # Combine the data into a 1 row dataframe and return
    df <- data.table(Article_ID, Title, Date, Abstract, Text, Source, Location, People, Organization, Type, Desk)
    return(df)
}

In [ ]:
df_list_nyt_1 <- lapply(NYT_1_xml, extract)
df_nyt_1 <- do.call(rbind, df_list_nyt_1)

df_list_nyt_2 <- lapply(NYT_2_xml, extract)
df_nyt_2 <- do.call(rbind, df_list_nyt_2)

df_list_nyt_3 <- lapply(NYT_3_xml, extract)
df_nyt_3 <- do.call(rbind, df_list_nyt_3)

saveRDS(df_nyt_1, "df_nyt_1.rds")
saveRDS(df_nyt_2, "df_nyt_2.rds")
saveRDS(df_nyt_3, "df_nyt_3.rds")

In [ ]:
df_list_wp_1 <- lapply(WP_1_xml, extract)
df_wp_1 <- do.call(rbind, df_list_wp_1)`

df_list_wp_2 <- lapply(WP_2_xml, extract)
df_wp_2 <- do.call(rbind, df_list_wp_2)

saveRDS(df_wp_1, "df_wp_1.rds")
saveRDS(df_wp_2, "df_wp_2.rds")

In [ ]:
# Take system time
start <- proc.time()

df_list_wsj <- lapply(WSJ_xml, extract)
df_wsj <- do.call(rbind, df_list_wsj)

# Show elapased time
difference <- proc.time() - start
print(paste("Creating the dataframe took:", format(difference[3]), "seconds."))

# Save files
saveRDS(df_wsj, "df_wsj.rds")

### Combine Dataset

In [ ]:
df_nyt_1 <- readRDS("df_nyt_1.rds")
df_nyt_2 <- readRDS("df_nyt_2.rds")
df_nyt_3 <- readRDS("df_nyt_3.rds")
df_wp_1 <- readRDS("df_wp_1.rds")
df_wp_2 <- readRDS("df_wp_2.rds")
df_wsj <- readRDS("df_wsj.rds")

In [ ]:
all_df <- bind_rows(df_nyt_1,
                    df_nyt_2,
                    df_nyt_3,
                    df_wp_1,
                    df_wp_2,
                    df_wsj)

In [ ]:
remove(df_nyt_1, df_nyt_2, df_nyt_3, df_wp_1, df_wp_2, df_wsj)

In [34]:
library(stringr)
all_df <- 
    all_df %>%
    filter(!str_detect(Type, "(Advertisement)|(Audio)|(Back Matter)|(Cartoon)|(General Information)|(Illustration)|(Image)|(Guideline)|(Front Matter)|(Undefined)"))

In [36]:
### Deduplicate
all_df_final <- all_df[, .(
    Title        = first(Title),
    Date         = first(Date),
    Abstract     = first(Abstract),
    Text         = first(Text),
    Source       = first(Source),
    Location     = paste(unique(Location[Location != ""]), collapse = "; "),
    People       = paste(unique(People[People != ""]),   collapse = "; "),
    Organization = paste(unique(Organization[Organization != ""]), collapse = "; "),
    Type         = paste(unique(Type[Type != ""]),       collapse = "; "),
    Desk         = paste(unique(Desk[Desk != ""]),       collapse = "; ")
), by = Article_ID]

In [37]:
saveRDS(all_df_final, "all_df_20260406.rds")

## Early Data

In [4]:
WP_Historical <- create_xml("WashingtonPost-HistoricalNewspaper")

[1] "Files in dataset 1015195"


In [8]:
batch_size <- 200000
n <- length(WP_Historical)

for (i in seq(1, n, by = batch_size)) {
  
  end_i <- min(i + batch_size - 1, n)
  
  cat("Processing", i, "to", end_i, "\n")
  
  batch_result <- lapply(WP_Historical[i:end_i], extract)
  batch_df <- bind_rows(batch_result)
  
  saveRDS(batch_df, 
          file = paste0("WP_Historical_batch_", i, "_", end_i, ".rds"))
  
  rm(batch_result, batch_df)
  gc()
}

Processing 1 to 2e+05 
Processing 200001 to 4e+05 
Processing 400001 to 6e+05 
Processing 600001 to 8e+05 
Processing 800001 to 1e+06 
Processing 1000001 to 1015195 


In [13]:
files <- list.files(pattern = "WP_Historical_batch_.*\\.rds$")
df <- bind_rows(lapply(files, readRDS))
saveRDS(df, "df_wp_historical.rds")

In [9]:
WSJ_Early_xml <- create_xml("WSJ1980-83")

[1] "Files in dataset 473250"


In [11]:
df_list_wsj <- lapply(WSJ_Early_xml, extract)
df_wsj <- do.call(rbind, df_list_wsj)

In [12]:
saveRDS(df_wsj, "df_wsj_historical.rds")

### Remove non-articles (e.g. advertisement)

In [18]:
library(stringr)
df_wp_clean <- 
    df %>%
    filter(!str_detect(Type, "(Advertisement)|(Birth Notice)|(Cartoon)|(General Information)|(Illustration)|(Image)|(Stock Quote)|(Front Matter)|(Undefined)"))

In [22]:
saveRDS(df_wp_clean, "df_wp_historical_clean.rds")

In [28]:
df_wsj_clean <- 
    df_wsj %>%
    filter(!str_detect(Type, "(Advertisement)|(Cartoon)|(Credit)|(General Information)|(Illustration)|(Front Matter)|(Undefined)"))

In [29]:
table(df_wsj_clean$Type)


                             Article                Commentary; Editorial 
                              138461                                 2382 
              Front Page/Cover Story Letter to the Editor; Correspondence 
                                5113                                  735 
                            Obituary 
                                   1 

In [30]:
saveRDS(df_wsj_clean, "df_wsj_historical_clean.rds")

### OCR

In [1]:
library(quanteda)

Package version: 4.3.1
Unicode version: 15.0
ICU version: 73.2

Parallel computing: disabled

See https://quanteda.io for tutorials and examples.



In [2]:
df_wsj_clean <- readRDS("df_wsj_historical_clean.rds")
df_wp_clean <- readRDS("df_wp_historical_clean.rds")

In [3]:
OCR_dict <- read.delim("MainDictionary.txt", 
                       header = FALSE, 
                       sep = "\t")

In [4]:
wsj_corpus <- corpus(df_wsj_clean, text_field = "Text")

### tokenize corpus removing unnecessary (i.e. semantically uninformative) elements
wsj_congress_toks <- tokens(wsj_corpus, 
                       remove_punct=T, 
                       remove_symbols=T, 
                       remove_numbers=T, 
                       remove_separators=T) %>%
                      tokens_tolower()

Warning message:
“NA is replaced by empty string”


In [ ]:
# Function to calculate percentage of words covered by the dictionary
calculate_coverage <- function(words, dictionary) {
  # Check which words are in the dictionary
  words_in_dictionary <- words %in% dictionary
  
  # Calculate the percentage
  percentage_covered <- mean(words_in_dictionary) * 100
  
  return(percentage_covered)
}

# Apply the function to each tokenized speech in the list
OCR_quality_wsj <- sapply(wsj_congress_toks, calculate_coverage, dictionary = OCR_dict$V1)

In [9]:
head(OCR_quality_wsj)

text1     text2     text3     text4     text5     text6 
 93.27177  93.36493  71.66667  92.20564  94.28571 100.00000

In [11]:
df_wsj_clean_OCR <-
  df_wsj_clean %>%
  mutate(percentage_covered = OCR_quality_wsj)

saveRDS(df_wsj_clean_OCR, "df_wsj_clean_OCR.rds")

In [15]:
wp_corpus <- corpus(df_wp_clean, text_field = "Text")

### tokenize corpus removing unnecessary (i.e. semantically uninformative) elements
wp_congress_toks <- tokens(wp_corpus, 
                       remove_punct=T, 
                       remove_symbols=T, 
                       remove_numbers=T, 
                       remove_separators=T) %>%
                      tokens_tolower()

Warning message:
“NA is replaced by empty string”


In [16]:
OCR_quality_wp <- sapply(wp_congress_toks, calculate_coverage, dictionary = OCR_dict$V1)

df_wp_clean_OCR <-
  df_wp_clean %>%
  mutate(percentage_covered = OCR_quality_wp)

saveRDS(df_wp_clean_OCR, "df_wp_clean_OCR.rds")